In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\viola\OneDrive\VIOLA\CS\Courses\AI\AI DEPI 2026\Graduation Project\DEPI-GRADUATION-PROJECT


In [ ]:
STAGE2_DIR = DATA_DIR / "processed" / "stage2"

TRAIN_PATH = STAGE2_DIR / "stage2_train_df.csv"
VAL_PATH = STAGE2_DIR / "stage2_val_df.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (5165, 60)
Validation shape: (1252, 60)


In [ ]:
required_columns = ["price", "model_key", "ad_id"]

for name, df in [("Train", train_df), ("Validation", val_df)]:
    print(name)

    for column in required_columns:
        print(f"{column}: {column in df.columns}")

    print(df[required_columns].isna().sum())

    print(
        "Unknown model keys:",
        df["model_key"].astype(str).str.lower().eq("unknown").sum()
    )

Train
price: True
model_key: True
ad_id: True
price        0
model_key    0
ad_id        0
dtype: int64
Unknown model keys: 465
Validation
price: True
model_key: True
ad_id: True
price        0
model_key    0
ad_id        0
dtype: int64
Unknown model keys: 122


In [ ]:
known_train = train_df[
    train_df["model_key"].notna()
    & train_df["model_key"].ne("unknown")
].copy()

model_stats = (
    known_train.groupby("model_key")["price"]
    .agg(
        model_median="median",
        group_size="count"
    )
)

model_stats["model_mad"] = (
    known_train.groupby("model_key")["price"]
    .apply(
        lambda x: (x - x.median()).abs().median()
    )
)

model_stats = model_stats.reset_index()

model_stats.head()

,model_key,model_median,group_size,model_mad
0,apple iphone 11,11500.0,242,1500.0
1,apple iphone 11 pro,13000.0,68,2000.0
2,apple iphone 11 pro max,16500.0,57,2000.0
3,apple iphone 12,14000.0,93,1700.0
4,apple iphone 12 mini,12000.0,15,1000.0


In [ ]:
print("Number of models:", len(model_stats))

print("\nGroup size distribution:")
print(model_stats["group_size"].describe())

print("\nModels by group size:")
print("1 listing:", (model_stats["group_size"] == 1).sum())
print("2 listings:", (model_stats["group_size"] == 2).sum())
print("3 or more:", (model_stats["group_size"] >= 3).sum())
print("5 or more:", (model_stats["group_size"] >= 5).sum())

print("\nModels with zero MAD:", (model_stats["model_mad"] == 0).sum())

Number of models: 722

Group size distribution:
count    722.000000
mean       6.509695
std       23.753719
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max      372.000000
Name: group_size, dtype: float64

Models by group size:
1 listing: 307
2 listings: 129
3 or more: 286
5 or more: 176

Models with zero MAD: 335


In [ ]:
mad_check = model_stats.assign(
    zero_mad=model_stats["model_mad"].eq(0)
)

print(
    pd.crosstab(
        mad_check["group_size"],
        mad_check["zero_mad"]
    ).head(10)
)

print("\nModels with zero MAD and at least 3 listings:")
print(
    (
        (model_stats["model_mad"] == 0)
        & (model_stats["group_size"] >= 3)
    ).sum()
)

zero_mad    False  True 
group_size              
1               0    307
2             115     14
3              45     11
4              54      0
5              25      3
6              27      0
7              18      0
8              17      0
9              12      0
10              7      0

Models with zero MAD and at least 3 listings:
14


In [ ]:
val_scored = val_df.merge(
    model_stats,
    on="model_key",
    how="left"
)

print("Validation rows:", len(val_scored))
print("Rows with model statistics:", val_scored["model_median"].notna().sum())
print("Rows without model statistics:", val_scored["model_median"].isna().sum())

print(
    "Rows with zero MAD:",
    val_scored["model_mad"].eq(0).sum()
)

print(
    "Rows with group size below 3:",
    val_scored["group_size"].lt(3).sum()
)

Validation rows: 1252
Rows with model statistics: 1015
Rows without model statistics: 237
Rows with zero MAD: 118
Rows with group size below 3: 165


In [ ]:
print("Validation model key status")

print(
    "Unknown:",
    val_scored["model_key"].astype(str).str.lower().eq("unknown").sum()
)

print(
    "No matching model statistics:",
    val_scored["model_median"].isna().sum()
)

print(
    "Group size = 1:",
    val_scored["group_size"].eq(1).sum()
)

print(
    "Group size = 2:",
    val_scored["group_size"].eq(2).sum()
)

print(
    "Group size >= 3 and MAD = 0:",
    (
        val_scored["group_size"].ge(3)
        & val_scored["model_mad"].eq(0)
    ).sum()
)

print(
    "Group size >= 3 and MAD > 0:",
    (
        val_scored["group_size"].ge(3)
        & val_scored["model_mad"].gt(0)
    ).sum()
)

Validation model key status
Unknown: 122
No matching model statistics: 237
Group size = 1: 111
Group size = 2: 54
Group size >= 3 and MAD = 0: 2
Group size >= 3 and MAD > 0: 848


In [ ]:
val_scored["price_ratio"] = np.nan
val_scored["modified_z_score"] = np.nan

has_median = val_scored["model_median"].notna()

valid_mad = (
    has_median
    & val_scored["group_size"].ge(3)
    & val_scored["model_mad"].gt(0)
)

val_scored.loc[has_median, "price_ratio"] = (
    val_scored.loc[has_median, "price"]
    / val_scored.loc[has_median, "model_median"]
)

val_scored.loc[valid_mad, "modified_z_score"] = (
    0.6745
    * (
        val_scored.loc[valid_mad, "price"]
        - val_scored.loc[valid_mad, "model_median"]
    )
    / val_scored.loc[valid_mad, "model_mad"]
)

print("Price ratio calculated:", val_scored["price_ratio"].notna().sum())
print("Modified Z-score calculated:", val_scored["modified_z_score"].notna().sum())

Price ratio calculated: 1015
Modified Z-score calculated: 848


In [ ]:
print(val_scored[["price_ratio", "modified_z_score"]].describe())

print("\nLowest price ratios:")
print(
    val_scored.nsmallest(5, "price_ratio")[
        ["ad_id", "model_key", "price", "model_median", "price_ratio"]
    ]
)

print("\nHighest price ratios:")
print(
    val_scored.nlargest(5, "price_ratio")[
        ["ad_id", "model_key", "price", "model_median", "price_ratio"]
    ]
)

       price_ratio  modified_z_score
count  1015.000000        848.000000
mean      1.032292         -2.713518
std       0.613983         70.547221
min       0.030120      -2022.825500
25%       0.857143         -0.674500
50%       1.000000          0.104387
75%       1.151087          0.809400
max      16.363636         94.992083

Lowest price ratios:
         ad_id                model_key   price  model_median  price_ratio
674  208981242  apple iphone 17 pro max  2500.0       83000.0     0.030120
263  209001279        samsung s26 ultra  2500.0       67000.0     0.037313
850  208971002          apple iphone 17  3000.0       56999.5     0.052632
607  208984704         oppo reno 15 pro  2000.0       37500.0     0.053333
31   209012037     apple iphone 14 plus  2000.0       37000.0     0.054054

Highest price ratios:
          ad_id        model_key    price  model_median  price_ratio
1030  208958544       realme c53  90000.0        5500.0    16.363636
762   208975182        xiaomi 15  

In [ ]:
review_columns = [
    "ad_id",
    "title",
    "model_key",
    "price",
    "model_median",
    "model_mad",
    "group_size",
    "price_ratio",
    "modified_z_score",
    "condition",
    "is_wanted_ad",
    "is_accessory_hint",
    "mentions_exchange"
]

review_columns = [
    col for col in review_columns
    if col in val_scored.columns
]

print("Lowest price ratios:")
display(
    val_scored.nsmallest(5, "price_ratio")[review_columns]
)

print("Highest price ratios:")
display(
    val_scored.nlargest(5, "price_ratio")[review_columns]
)

Lowest price ratios:


,ad_id,title,model_key,price,model_median,model_mad,group_size,price_ratio,modified_z_score,condition,is_wanted_ad,is_accessory_hint,mentions_exchange
674,208981242,ايفون 17 برو ماكس حجم صغير,apple iphone 17 pro max,2500.0,83000.0,7000.0,342.0,0.030120,-7.756750,NaN,False,False,False
263,209001279,المدارس فتحت ووقت التغيير جه مع سامسونج S26 الترا,samsung s26 ultra,2500.0,67000.0,2000.0,9.0,0.037313,-21.752625,NaN,False,False,False
850,208971002,آبل - آيفون 17 اير ميني هاي كوبي,apple iphone 17,3000.0,56999.5,11000.0,74.0,0.052632,-3.311151,NaN,False,False,False
607,208984704,طب شوف OPPO Reno 15 Pro الأول! الامكانيات ب...,oppo reno 15 pro,2000.0,37500.0,501.0,9.0,0.053333,-47.793912,NaN,False,False,False
31,209012037,آبل - آيفون 14 بلس,apple iphone 14 plus,2000.0,37000.0,2000.0,2.0,0.054054,NaN,NaN,False,False,False


Highest price ratios:


,ad_id,title,model_key,price,model_median,model_mad,group_size,price_ratio,modified_z_score,condition,is_wanted_ad,is_accessory_hint,mentions_exchange
1030,208958544,ريلمى c53 nfc 256,realme c53,90000.0,5500.0,600.0,14.0,16.363636,94.992083,NaN,False,False,False
762,208975182,شاومي 15,xiaomi 15,90001.0,13250.0,4500.0,8.0,6.792528,11.504122,NaN,False,False,False
223,209003453,سامسونج زد فليب 4,samsung z flip4,20000.0,5000.0,0.0,1.0,4.000000,NaN,NaN,False,False,False
163,209006473,xiaomi 11 lite 5g للبدل,xiaomi 11 lite,8500.0,2500.0,0.0,1.0,3.400000,NaN,NaN,False,False,True
98,209009089,شاومي 11 لايت,xiaomi 11 lite,8000.0,2500.0,0.0,1.0,3.200000,NaN,NaN,False,False,False


In [ ]:
val_scored["reference_status"] = "ready_for_scoring"

unknown_model = (
    val_scored["model_key"].astype(str).str.lower().eq("unknown")
)

no_reference = val_scored["model_median"].isna()

zero_mad = (
    val_scored["model_median"].notna()
    & val_scored["model_mad"].eq(0)
)

small_group = (
    val_scored["group_size"].notna()
    & val_scored["group_size"].lt(3)
)

val_scored.loc[small_group, "reference_status"] = "small_group"
val_scored.loc[zero_mad, "reference_status"] = "zero_mad"
val_scored.loc[no_reference, "reference_status"] = "no_train_reference"
val_scored.loc[unknown_model, "reference_status"] = "unknown_model"

print(val_scored["reference_status"].value_counts())

reference_status
ready_for_scoring     848
unknown_model         122
zero_mad              118
no_train_reference    115
small_group            49
Name: count, dtype: int64


In [ ]:
ready_cases = val_scored[
    val_scored["reference_status"] == "ready_for_scoring"
].copy()

print("Ready cases:", len(ready_cases))

print("\nModified Z-score distribution:")
print(ready_cases["modified_z_score"].describe())

print("\nPrice ratio distribution:")
print(ready_cases["price_ratio"].describe())

Ready cases: 848

Modified Z-score distribution:
count     848.000000
mean       -2.713518
std        70.547221
min     -2022.825500
25%        -0.674500
50%         0.104387
75%         0.809400
max        94.992083
Name: modified_z_score, dtype: float64

Price ratio distribution:
count    848.000000
mean       1.038994
std        0.627035
min        0.030120
25%        0.882353
50%        1.011053
75%        1.164680
max       16.363636
Name: price_ratio, dtype: float64


In [ ]:
extreme_cases = ready_cases.copy()

print("Lowest Modified Z-scores:")
print(
    extreme_cases.nsmallest(10, "modified_z_score")[
        [
            "ad_id",
            "title",
            "model_key",
            "price",
            "model_median",
            "model_mad",
            "group_size",
            "price_ratio",
            "modified_z_score"
        ]
    ].to_string(index=False)
)

print("\nHighest Modified Z-scores:")
print(
    extreme_cases.nlargest(10, "modified_z_score")[
        [
            "ad_id",
            "title",
            "model_key",
            "price",
            "model_median",
            "model_mad",
            "group_size",
            "price_ratio",
            "modified_z_score"
        ]
    ].to_string(index=False)
)

Lowest Modified Z-scores:
    ad_id                                                      title               model_key   price  model_median  model_mad  group_size  price_ratio  modified_z_score
208991118                             إنفنيكس هوت 50 حاله ممتازه جدا          infinix hot 50  4500.0        7499.0        1.0         7.0     0.600080      -2022.825500
207554617                                        انفنكس هوت 50 للبدل          infinix hot 50  7000.0        7499.0        1.0         7.0     0.933458       -336.575500
208984704 طب شوف OPPO Reno 15 Pro الأول!    الامكانيات بتاعتوو توحفه        oppo reno 15 pro  2000.0       37500.0      501.0         9.0     0.053333        -47.793912
209001279          المدارس فتحت ووقت التغيير جه مع سامسونج S26 الترا       samsung s26 ultra  2500.0       67000.0     2000.0         9.0     0.037313        -21.752625
208687482                                        سامسونج اس 25 الترا       samsung s25 ultra  6250.0       55000.0     3000.0    

In [ ]:
infinix_prices = known_train[
    known_train["model_key"] == "infinix hot 50"
][["ad_id", "title", "price"]].sort_values("price")

print(infinix_prices.to_string(index=False))

print("\nPrice statistics:")
print(infinix_prices["price"].describe())

print("\nAbsolute deviations from median:")
print(
    (infinix_prices["price"] - infinix_prices["price"].median())
    .abs()
    .sort_values()
    .to_list()
)

    ad_id          title  price
209006023 إنفنيكس هوت 50 5000.0
208952772 إنفنيكس هوت 50 6500.0
208986820 إنفنيكس هوت 50 6800.0
209006876 Infinix Hot 50 7499.0
208997369 إنفنيكس هوت 50 7500.0
208954947 إنفنيكس هوت 50 7500.0
208950683 إنفنيكس هوت 50 7500.0

Price statistics:
count       7.000000
mean     6899.857143
std       932.630764
min      5000.000000
25%      6650.000000
50%      7499.000000
75%      7500.000000
max      7500.000000
Name: price, dtype: float64

Absolute deviations from median:
[0.0, 1.0, 1.0, 1.0, 699.0, 999.0, 2499.0]


In [ ]:
small_mad_models = model_stats[
    (model_stats["group_size"] >= 3)
    & (model_stats["model_mad"] <= 1)
].sort_values(
    ["model_mad", "group_size"],
    ascending=[True, False]
)

print("Models with group size >= 3 and MAD <= 1:")
print("Number of models:", len(small_mad_models))

print(
    small_mad_models[
        ["model_key", "group_size", "model_median", "model_mad"]
    ].to_string(index=False)
)

Models with group size >= 3 and MAD <= 1:
Number of models: 15
        model_key  group_size  model_median  model_mad
    honor 600 pro           5       48000.0        0.0
honor magic 5 pro           5       21000.0        0.0
      samsung m31           5        6000.0        0.0
        honor x7c           3        8000.0        0.0
  huawei y7 prime           3        2200.0        0.0
  infinix note 11           3        4500.0        0.0
  infinix smart 6           3        2100.0        0.0
motorola moto g35           3        8500.0        0.0
         oppo a53           3        3500.0        0.0
       oppo a5pro           3       12000.0        0.0
         oppo a96           3        5000.0        0.0
 oppo reno 13f 5g           3       15000.0        0.0
    realme 16 pro           3       25000.0        0.0
     vivo x200 fe           3       34000.0        0.0
   infinix hot 50           7        7499.0        1.0


In [ ]:
zero_mad_models = model_stats[
    (model_stats["group_size"] >= 3)
    & (model_stats["model_mad"] == 0)
]["model_key"]

for model in zero_mad_models:
    prices = known_train.loc[
        known_train["model_key"] == model,
        "price"
    ]

    print(f"\n{model}")
    print("Prices:", sorted(prices.tolist()))
    print("Unique prices:", prices.nunique())


honor 600 pro
Prices: [48000.0, 48000.0, 48000.0, 48000.0, 50000.0]
Unique prices: 2

honor magic 5 pro
Prices: [19500.0, 21000.0, 21000.0, 21000.0, 24000.0]
Unique prices: 3

honor x7c
Prices: [8000.0, 8000.0, 9000.0]
Unique prices: 2

huawei y7 prime
Prices: [2200.0, 2200.0, 2500.0]
Unique prices: 2

infinix note 11
Prices: [4000.0, 4500.0, 4500.0]
Unique prices: 2

infinix smart 6
Prices: [2100.0, 2100.0, 3000.0]
Unique prices: 2

motorola moto g35
Prices: [7500.0, 8500.0, 8500.0]
Unique prices: 2

oppo a53
Prices: [3500.0, 3500.0, 4000.0]
Unique prices: 2

oppo a5pro
Prices: [12000.0, 12000.0, 12000.0]
Unique prices: 1

oppo a96
Prices: [5000.0, 5000.0, 8000.0]
Unique prices: 2

oppo reno 13f 5g
Prices: [14000.0, 15000.0, 15000.0]
Unique prices: 2

realme 16 pro
Prices: [25000.0, 25000.0, 30000.0]
Unique prices: 2

samsung m31
Prices: [4000.0, 5500.0, 6000.0, 6000.0, 6000.0]
Unique prices: 3

vivo x200 fe
Prices: [33000.0, 34000.0, 34000.0]
Unique prices: 2


In [ ]:
zero_mad_details = (
    known_train[
        known_train["model_key"].isin(zero_mad_models)
    ]
    .groupby("model_key")["price"]
    .agg(
        group_size="count",
        unique_prices="nunique",
        min_price="min",
        max_price="max",
        median_price="median",
        median_count=lambda x: (x == x.median()).sum()
    )
    .reset_index()
)

zero_mad_details["median_share"] = (
    zero_mad_details["median_count"]
    / zero_mad_details["group_size"]
)

zero_mad_details.sort_values(
    ["group_size", "median_share"],
    ascending=[False, False]
)

,model_key,group_size,unique_prices,min_price,max_price,median_price,median_count,median_share
0,honor 600 pro,5,2,48000.0,50000.0,48000.0,4,0.800000
1,honor magic 5 pro,5,3,19500.0,24000.0,21000.0,3,0.600000
12,samsung m31,5,3,4000.0,6000.0,6000.0,3,0.600000
8,oppo a5pro,3,1,12000.0,12000.0,12000.0,3,1.000000
2,honor x7c,3,2,8000.0,9000.0,8000.0,2,0.666667
3,huawei y7 prime,3,2,2200.0,2500.0,2200.0,2,0.666667
4,infinix note 11,3,2,4000.0,4500.0,4500.0,2,0.666667
5,infinix smart 6,3,2,2100.0,3000.0,2100.0,2,0.666667
6,motorola moto g35,3,2,7500.0,8500.0,8500.0,2,0.666667
7,oppo a53,3,2,3500.0,4000.0,3500.0,2,0.666667


In [ ]:
model_stats["mad_to_median_ratio"] = (
    model_stats["model_mad"] / model_stats["model_median"]
)

small_mad_check = model_stats[
    (model_stats["group_size"] >= 3)
    & (model_stats["model_mad"] > 0)
].sort_values("mad_to_median_ratio")

print(
    small_mad_check[
        [
            "model_key",
            "group_size",
            "model_median",
            "model_mad",
            "mad_to_median_ratio"
        ]
    ].head(15).to_string(index=False)
)

              model_key  group_size  model_median  model_mad  mad_to_median_ratio
         infinix hot 50           7        7499.0        1.0             0.000133
            honor robot           3       94000.0     1000.0             0.010638
        xiaomi redmi a3           4        4500.0       50.0             0.011111
           oppo find x5           3       16700.0      200.0             0.011976
xiaomi redmi note 9 pro           4        3850.0       50.0             0.012987
         xiaomi poco f7           4       19000.0      250.0             0.013158
       oppo reno 15 pro           9       37500.0      501.0             0.013360
            samsung s22           4       16750.0      250.0             0.014925
             xiaomi 17t           4       32750.0      500.0             0.015267
            samsung s26           6       46650.0      750.0             0.016077
xiaomi redmi note 15 5g           4       14250.0      250.0             0.017544
      motorola m

In [ ]:
print("Group size distribution:")

print(
    model_stats["group_size"]
    .value_counts()
    .sort_index()
    .head(15)
)

print("\nNumber of models by minimum group size:")

for minimum_size in [2, 3, 4, 5, 10]:
    print(
        f"At least {minimum_size}:",
        (model_stats["group_size"] >= minimum_size).sum()
    )

Group size distribution:
group_size
1     307
2     129
3      56
4      54
5      28
6      27
7      18
8      17
9      12
10      7
11      6
12      4
13      3
14      4
15      4
Name: count, dtype: int64

Number of models by minimum group size:
At least 2: 415
At least 3: 286
At least 4: 230
At least 5: 176
At least 10: 74


In [ ]:
print("Validation rows eligible by minimum group size:")

for minimum_size in [2, 3, 4, 5, 10]:
    eligible = (
        val_scored["model_median"].notna()
        & val_scored["group_size"].ge(minimum_size)
    )

    print(
        f"At least {minimum_size}:",
        eligible.sum(),
        f"({eligible.mean():.1%})"
    )

Validation rows eligible by minimum group size:
At least 2: 904 (72.2%)
At least 3: 850 (67.9%)
At least 4: 812 (64.9%)
At least 5: 750 (59.9%)
At least 10: 615 (49.1%)
